# Week 2 Day 4 — CrewAI: Multi-Agent Collaboration, Roles & Task Delegation

**Business task chosen:** Review a sales dataset, generate business insights, and write a stakeholder-ready summary.

This mirrors the kind of task a data team hands off across specialists: one person cleans/profiles the data, one person turns numbers into insights, and one person writes it up for a non-technical audience.

**Deliverable covers:**
- Task 1 — Multi-agent design thinking (roles, goals, backstories)
- Task 2 — Agents + role-appropriate tools
- Task 3 — Tasks + sequential Process, execution log review
- Task 4 — Hierarchical Process with a manager agent
- Task 5 — Token/cost logging + evaluation scoring

**Setup (run once in terminal, not in a cell):**
```
pip install crewai crewai-tools --break-system-packages
```


In [15]:
import os

# Jupyter's kernel runs its own asyncio event loop in the background.
# Newer crewai versions make internal async LLM calls, and refuse to run
# synchronously (crew.kickoff()) inside an already-running loop, raising:
# "Agent execution was invoked synchronously from within a running event loop".
# nest_asyncio patches the loop so nested sync calls work fine in Jupyter.
# (pip install nest_asyncio --break-system-packages  if not already installed)
import nest_asyncio
nest_asyncio.apply()




In [2]:
import time
import json
from textwrap import dedent

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import FileReadTool


## LLM config — Gemini

CrewAI's tools (`FileReadTool`/`CSVSearchTool`) use OpenAI embeddings by default for their internal RAG search. Since we're on Gemini, we also point embeddings at Gemini so no OpenAI key is needed anywhere.

In [3]:
GEMINI_MODEL = "gemini/gemini-3.5-flash-lite"  # swap for "gemini/gemini-1.5-pro" etc. if needed

gemini_llm = LLM(
    model=GEMINI_MODEL,
    api_key=os.environ.get("GEMINI_API_KEY"),
    temperature=0.3,
)

# Note: crewai_tools' *SearchTool RAG tools (e.g. CSVSearchTool) currently
# have an open bug where their internal validator requires OPENAI_API_KEY
# to be set even when llm/embedder config points elsewhere (see
# github.com/crewAIInc/crewAI issues #4028, #2850, #2517). Since our sample
# CSV is tiny (9 rows), we avoid that whole RAG/embedding subsystem and use
# only FileReadTool below, which reads the raw file directly, no OpenAI
# key required.


## 0. Sample data

So the notebook is runnable end-to-end without an external file. Replace with your own dataset path if you have one.

In [4]:
SAMPLE_CSV_PATH = "sales_data.csv"

def write_sample_dataset():
    import csv
    rows = [
        ["month", "region", "product", "units_sold", "revenue"],
        ["Jan", "North", "Widget-A", 120, 6000],
        ["Feb", "North", "Widget-A", 90, 4500],
        ["Mar", "North", "Widget-A", 60, 3000],
        ["Jan", "South", "Widget-A", 80, 4000],
        ["Feb", "South", "Widget-A", 95, 4750],
        ["Mar", "South", "Widget-A", 130, 6500],
        ["Jan", "North", "Widget-B", 40, 3200],
        ["Feb", "North", "Widget-B", 55, 4400],
        ["Mar", "North", "Widget-B", 70, 5600],
    ]
    with open(SAMPLE_CSV_PATH, "w", newline="") as f:
        csv.writer(f).writerows(rows)

write_sample_dataset()
print("Sample dataset written to", SAMPLE_CSV_PATH)


Sample dataset written to sales_data.csv


## Task 1 — Multi-Agent Design Thinking

**Why 3 specialists instead of 1 generalist here?**
A single generalist agent has to hold "clean/profile the data", "form a business narrative", and "write for a non-technical exec" all in one prompt/context — these pull the model toward different tones and priorities, and a long instruction list increases the chance one step gets skipped or diluted. Splitting into agents gives each step a tight, role-scoped prompt (better instruction-following) and lets you swap or re-run just the failing step instead of the whole chain.

**Where it ISN'T worth it:** trivial datasets (a handful of rows/columns) or one-off exploratory questions — the coordination/token overhead of 3 agents talking to each other costs more than it saves versus one well-prompted agent doing everything in a single pass.

### Roles
- **Data Analyst** — reads the raw data, produces a verified statistical profile.
- **Business Insight Generator** — interprets the profile into business-relevant insights.
- **Stakeholder Report Writer** — turns insights into a concise, non-technical summary.


## Task 2 — Build Agents & Assign Tools

Tool access is kept role-appropriate:
- **Data Analyst**: `FileReadTool` only — needs to actually read the raw data. No other agent gets file access, so they only ever work from the analyst's verified output (prevents them silently re-deriving different numbers). (We dropped `CSVSearchTool`/RAG-based search here: for a 9-row sample file it adds no value, and it currently has an open crewai_tools bug requiring an OpenAI key even when configured for another provider — see note in the LLM config cell above.)
- **Insight Generator**: no tools — works only from the analyst's profile.
- **Report Writer**: no tools — pure writing task from the insight list.


In [5]:
def build_agents(manager: bool = False):
    """Builds the 3 specialist agents. If manager=True, also returns a
    manager agent for the hierarchical process (Task 4)."""

    file_tool = FileReadTool(file_path=SAMPLE_CSV_PATH)

    # --- Agent 1: Data Analyst ---
    data_analyst = Agent(
        role="Data Analyst",
        goal=(
            "Read the sales dataset and produce an accurate, well-structured "
            "statistical profile: totals, trends by month/region/product, "
            "and any notable anomalies."
        ),
        backstory=dedent("""
            You are a meticulous data analyst who has spent years turning raw
            spreadsheets into clean, trustworthy numbers. You never guess at
            a figure you haven't verified from the data itself, and you flag
            gaps or anomalies rather than smoothing them over.
        """),
        tools=[file_tool],
        llm=gemini_llm,
        verbose=True,
        allow_delegation=False,
    )

    # --- Agent 2: Insight Generator ---
    insight_generator = Agent(
        role="Business Insight Generator",
        goal=(
            "Interpret the analyst's statistical profile and surface the "
            "3-5 most business-relevant insights, including likely causes "
            "and what they imply for decision-making."
        ),
        backstory=dedent("""
            You are a business analyst who translates numbers into "so
            what". You've sat in enough stakeholder meetings to know execs
            care about implications and next steps, not raw statistics.
        """),
        tools=[],
        llm=gemini_llm,
        verbose=True,
        allow_delegation=False,
    )

    # --- Agent 3: Report Writer ---
    report_writer = Agent(
        role="Stakeholder Report Writer",
        goal=(
            "Turn the insights into a concise, non-technical, "
            "stakeholder-ready summary (under 300 words) with a clear "
            "headline takeaway and 2-3 recommended actions."
        ),
        backstory=dedent("""
            You are a communications specialist who writes executive
            summaries for people with no time and no patience for jargon.
            You lead with the takeaway, not the methodology.
        """),
        tools=[],
        llm=gemini_llm,
        verbose=True,
        allow_delegation=False,
    )

    agents = {
        "analyst": data_analyst,
        "insights": insight_generator,
        "writer": report_writer,
    }

    if manager:
        manager_agent = Agent(
            role="Analytics Team Manager",
            goal=(
                "Coordinate the analyst, insight generator, and writer to "
                "produce an accurate, well-written stakeholder summary, "
                "reviewing each hand-off for quality before passing it on."
            ),
            backstory=dedent("""
                You lead a small analytics team. You don't do the analysis
                yourself — you delegate to the right specialist, check
                their work meets the brief, and send it back for revision
                if it doesn't.
            """),
            tools=[],
            llm=gemini_llm,
            verbose=True,
            allow_delegation=True,
        )
        agents["manager"] = manager_agent

    return agents


## Task 3 — Define Tasks & Process (sequential)

Each `Task` has a `context` pointing to the previous task, so outputs chain through the crew.

> **Format-mismatch note:** in an early run, the analyst's raw text output included a leading sentence like "Here is the profile:" before the JSON, which broke the insight generator's parsing expectations downstream. Fix: tightened `expected_output` to say *"Return ONLY the JSON, no prose around it"* — the standard CrewAI fix pattern is to push formatting constraints into `expected_output` rather than `description`, since that's what the agent self-checks against.

In [6]:
def build_tasks(agents):
    task_profile = Task(
        description=dedent("""
            Read {csv_path} and produce a statistical profile of the sales
            data: total revenue and units by month, by region, and by
            product. Note the single biggest month-over-month change you
            find and name the region/product it belongs to.
        """).format(csv_path=SAMPLE_CSV_PATH),
        expected_output=dedent("""
            A JSON object with keys: "by_month", "by_region", "by_product"
            (each a dict of totals), and "biggest_change" (a one-sentence
            description). Return ONLY the JSON, no prose around it.
        """),
        agent=agents["analyst"],
    )

    task_insights = Task(
        description=dedent("""
            Using the analyst's JSON profile, identify the 3-5 most
            business-relevant insights. For each, state the finding and a
            plausible business reason for it.
        """),
        expected_output=(
            "A numbered list of 3-5 insights, each 1-2 sentences, in plain "
            "English (not JSON)."
        ),
        agent=agents["insights"],
        context=[task_profile],
    )

    task_report = Task(
        description=dedent("""
            Write a stakeholder-ready summary based on the insights list.
            Lead with a one-line headline takeaway, then 2-3 short
            paragraphs, then 2-3 recommended actions as bullet points.
            Keep the whole thing under 300 words and avoid jargon.
        """),
        expected_output=(
            "A markdown-formatted executive summary: headline, body, "
            "and a 'Recommended actions' bullet list."
        ),
        agent=agents["writer"],
        context=[task_insights],
    )

    return [task_profile, task_insights, task_report]


### Run the sequential crew

This cell actually calls the Gemini API and will show the full execution log below (agent thoughts, tool calls, hand-offs) — this **is** your "capture and review the full execution log" deliverable for Task 3.

In [7]:
async def run_sequential():
    print("\n" + "=" * 70)
    print("SEQUENTIAL RUN")
    print("=" * 70)

    agents = build_agents(manager=False)
    tasks = build_tasks(agents)

    crew = Crew(
        agents=[agents["analyst"], agents["insights"], agents["writer"]],
        tasks=tasks,
        process=Process.sequential,
        verbose=True,
    )

    start = time.time()
    # Jupyter's kernel already runs an event loop, and crewai's sync
    # kickoff() deliberately refuses to run inside one (it returns a
    # coroutine and raises rather than block). Use kickoff_async() +
    # top-level await, which Jupyter cells support directly.
    result = await crew.kickoff_async()
    elapsed = time.time() - start

    return result, elapsed, crew

seq_result, seq_time, seq_crew = await run_sequential()
print("\nSEQUENTIAL FINAL OUTPUT:\n", seq_result)



SEQUENTIAL RUN


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5ac021b8-e0d5-4dfc-b11b-8b7464d63e58                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Read sales_data.csv and produce a statistical profile of the sales                                             │
│  data: total revenue and units by month, by region, and by                                                      │
│  product. Note the single biggest month-over-month change you                                                   │
│  find and name the region/product it belongs to.                                                                │
│                                                                                                                 │
│  ID: 19bb3318-7740-40f2-aa78-3293edc0a42a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Analyst                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Read sales_data.csv and produce a statistical profile of the sales                                             │
│  data: total revenue and units by month, by region, and by                                                      │
│  product. Note the single biggest month-over-month change you                                                   │
│  find and name the region/product it belongs to.                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_a_files_content                                                                                     │
│  Args: {'start_line': 1, 'line_count': None, 'file_path': 'sales_data.csv'}                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stakeholder Report Writer                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Takeaway: Overall revenue is growing despite a heavy reliance on a single product and a lagging Southern     │
│  region.**                                                                                                      │
│                                                                                                                 │
│  Our business is on solid ground. Overall revenue grew steadily over the past quarter, proving that our         │
│  general momentum remains strong even after a temporary sales dip in February.                                  │
│                                                                                                                 │
│  However, our success is currently built on an imbalance. Widget-A is our undeniable powerhouse, driving 77%    │
│  of all units sold. While this shows strong customer demand, it leaves us vulnerable if market tastes change.   │
│  At the same time, our geographic performance is split: the North region dramatically outperforms the South,    │
│  bringing in 58% more revenue.                                                                                  │
│                                                                                                                 │
│  On the bright side, our lower-volume product, Widget-B, commands a premium price tag of $80 per unit compared  │
│  to Widget-A's $50. This gives us a clear runway to boost profits by increasing Widget-B's sales volume         │
│  without dropping its price.                                                                                    │
│                                                                                                                 │
│  **Recommended actions:**                                                                                       │
│  * **Investigate the North-South gap:** Send a team to analyze why the North is outperforming the South so we   │
│  can apply those winning sales tactics to the Southern region.                                                  │
│  * **Scale Widget-B marketing:** Increase promotional efforts for Widget-B to drive higher sales volume and     │
│  capitalize on its strong $80 price point.                                                                      │
│  * **Protect against single-product risk:** Begin exploring ways to diversify our product lineup or add         │
│  features to Widget-A to protect our market share against future shifts.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Write a stakeholder-ready summary based on the insights list.                                                  │
│  Lead with a one-line headline takeaway, then 2-3 short                                                         │
│  paragraphs, then 2-3 recommended actions as bullet points.                                                     │
│  Keep the whole thing under 300 words and avoid jargon.                                                         │
│                                                                                                                 │
│  Agent: Stakeholder Report Writer                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SEQUENTIAL FINAL OUTPUT:
 **Takeaway: Overall revenue is growing despite a heavy reliance on a single product and a lagging Southern region.**

Our business is on solid ground. Overall revenue grew steadily over the past quarter, proving that our general momentum remains strong even after a temporary sales dip in February. 

However, our success is currently built on an imbalance. Widget-A is our undeniable powerhouse, driving 77% of all units sold. While this shows strong customer demand, it leaves us vulnerable if market tastes change. At the same time, our geographic performance is split: the North region dramatically outperforms the South, bringing in 58% more revenue. 

On the bright side, our lower-volume product, Widget-B, commands a premium price tag of $80 per unit compared to Widget-A's $50. This gives us a clear runway to boost profits by increasing Widget-B's sales volume without dropping its price.

**Recommended actions:**
* **Investigate the North-South gap:** Send a te

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5ac021b8-e0d5-4dfc-b11b-8b7464d63e58                                                                       │
│  Final Output: **Takeaway: Overall revenue is growing despite a heavy reliance on a single product and a        │
│  lagging Southern region.**                                                                                     │
│                                                                                                                 │
│  Our business is on solid ground. Overall revenue grew steadily over the past quarter, proving that our         │
│  general momentum remains strong even after a temporary sales dip in February.                                  │
│                                                                                                                 │
│  However, our success is currently built on an imbalance. Widget-A is our undeniable powerhouse, driving 77%    │
│  of all units sold. While this shows strong customer demand, it leaves us vulnerable if market tastes change.   │
│  At the same time, our geographic performance is split: the North region dramatically outperforms the South,    │
│  bringing in 58% more revenue.                                                                                  │
│                                                                                                                 │
│  On the bright side, our lower-volume product, Widget-B, commands a premium price tag of $80 per unit compared  │
│  to Widget-A's $50. This gives us a clear runway to boost profits by increasing Widget-B's sales volume         │
│  without dropping its price.                                                                                    │
│                                                                                                                 │
│  **Recommended actions:**                                                                                       │
│  * **Investigate the North-South gap:** Send a team to analyze why the North is outperforming the South so we   │
│  can apply those winning sales tactics to the Southern region.                                                  │
│  * **Scale Widget-B marketing:** Increase promotional efforts for Widget-B to drive higher sales volume and     │
│  capitalize on its strong $80 price point.                                                                      │
│  * **Protect against single-product risk:** Begin exploring ways to diversify our product lineup or add         │
│  features to Widget-A to protect our market share against future shifts.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Task 4 — Hierarchical Delegation

Same crew, rebuilt with a manager agent using `Process.hierarchical`. The manager delegates to and reviews each specialist's work before passing it on.

In [ ]:
async def run_hierarchical():
    print("\n" + "=" * 70)
    print("HIERARCHICAL RUN")
    print("=" * 70)

    agents = build_agents(manager=True)
    tasks = build_tasks(agents)

    crew = Crew(
        agents=[agents["analyst"], agents["insights"], agents["writer"]],
        tasks=tasks,
        process=Process.hierarchical,
        manager_agent=agents["manager"],
        verbose=True,
    )

    start = time.time()
    result = await crew.kickoff_async()
    elapsed = time.time() - start

    return result, elapsed, crew

hier_result, hier_time, hier_crew = await run_hierarchical()
print("\nHIERARCHICAL FINAL OUTPUT:\n", hier_result)



HIERARCHICAL RUN


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 037c1c4f-f758-47e5-a09f-6041b8c72295                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Read sales_data.csv and produce a statistical profile of the sales                                             │
│  data: total revenue and units by month, by region, and by                                                      │
│  product. Note the single biggest month-over-month change you                                                   │
│  find and name the region/product it belongs to.                                                                │
│                                                                                                                 │
│  ID: 217e6356-b828-4fe7-ac48-837eb76bbc90                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analytics Team Manager                                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Read sales_data.csv and produce a statistical profile of the sales                                             │
│  data: total revenue and units by month, by region, and by                                                      │
│  product. Note the single biggest month-over-month change you                                                   │
│  find and name the region/product it belongs to.                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'We need to read sales_data.csv and produce:\n1. Total revenue and units by month\n2. Total  │
│  revenue and units by region\n3. Total revenue and units by product\n4. The single biggest month-...            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Read sales_data.csv and produce a statistical profile of the sales                                             │
│  data: total revenue and units by month, by region, and by                                                      │
│  product. Note the single biggest month-over-month change you                                                   │
│  find and name the region/product it belongs to.                                                                │
│                                                                                                                 │
│  Agent: Analytics Team Manager                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analytics Team Manager                                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Using the analyst's JSON profile, identify the 3-5 most                                                        │
│  business-relevant insights. For each, state the finding and a                                                  │
│  plausible business reason for it.                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Using the analyst's JSON profile, identify the 3-5 most                                                        │
│  business-relevant insights. For each, state the finding and a                                                  │
│  plausible business reason for it.                                                                              │
│                                                                                                                 │
│  ID: 3e090598-6c35-489f-bb5d-de1b659f0da0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'coworker': 'Business Insight Generator', 'context': 'Here is the analyst\'s JSON profile containing    │
│  monthly revenue and units, regional breakdown, product breakdown, and the biggest month-over-month...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Insight Generator                                                                              │
│                                                                                                                 │
│  Task: Identify 3-5 most business-relevant insights with findings and plausible business reasons.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Task 5 — Evaluation & Cost Awareness

Logs token usage/approximate cost per run, and gives you a manual scoring sheet against 3 success criteria.

In [10]:
def get_token_usage(crew):
    """CrewAI exposes usage_metrics on the crew after kickoff (varies by
    version — fall back gracefully if unavailable)."""
    try:
        return crew.usage_metrics
    except Exception:
        return None


def approx_cost(usage, price_per_1k_input=0.0005, price_per_1k_output=0.0015):
    """Rough cost estimate — adjust price_per_1k_* to your actual model pricing."""
    if not usage:
        return None
    prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
    completion_tokens = getattr(usage, "completion_tokens", 0) or 0
    cost = (prompt_tokens / 1000) * price_per_1k_input + \
           (completion_tokens / 1000) * price_per_1k_output
    return round(cost, 5)


SUCCESS_CRITERIA = [
    "Factual grounding — every number in the final report traces back to the analyst's JSON profile.",
    "Completeness — report covers month, region, and product trends, not just one dimension.",
    "Tone — report reads as non-technical/executive-friendly, under 300 words.",
]


def score_run(label, result_text):
    """Manual scoring placeholder — fill in 1-5 per criterion after reading
    the actual output text above."""
    print(f"\n--- Manual scoring sheet: {label} ---")
    for c in SUCCESS_CRITERIA:
        print(f"  [ ] {c}  -> score: __ /5")
    print("  (fill in after reading the printed result above)")


def evaluate_run(label, result, elapsed, crew):
    usage = get_token_usage(crew)
    cost = approx_cost(usage)
    print(f"\n--- {label} metrics ---")
    print(f"Wall-clock time: {elapsed:.1f}s")
    print(f"Token usage: {usage}")
    print(f"Approx. cost: ${cost}" if cost is not None else "Approx. cost: n/a")
    score_run(label, str(result))
    return {"label": label, "elapsed": elapsed, "usage": str(usage), "cost": cost}


seq_metrics = evaluate_run("Sequential", seq_result, seq_time, seq_crew)
hier_metrics = evaluate_run("Hierarchical", hier_result, hier_time, hier_crew)
print(json.dumps({"sequential": seq_metrics, "hierarchical": hier_metrics}, indent=2))



--- Sequential metrics ---
Wall-clock time: 50.6s
Token usage: total_tokens=9825 prompt_tokens=6918 cached_prompt_tokens=0 completion_tokens=2907 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=12
Approx. cost: $0.00782

--- Manual scoring sheet: Sequential ---
  [ ] Factual grounding — every number in the final report traces back to the analyst's JSON profile.  -> score: __ /5
  [ ] Completeness — report covers month, region, and product trends, not just one dimension.  -> score: __ /5
  [ ] Tone — report reads as non-technical/executive-friendly, under 300 words.  -> score: __ /5
  (fill in after reading the printed result above)

--- Hierarchical metrics ---
Wall-clock time: 373.9s
Token usage: total_tokens=170096 prompt_tokens=136808 cached_prompt_tokens=0 completion_tokens=33288 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=112
Approx. cost: $0.11834

--- Manual scoring sheet: Hierarchical ---
  [ ] Factual grounding — every number in the final rep

### Sequential vs Hierarchical — comparison table

| Aspect | Sequential | Hierarchical |
|---|---|---|
| Quality | Consistent, predictable hand-offs | Can catch/fix a bad hand-off via manager review |
| Latency / cost | Lower — no manager overhead | Higher — manager adds extra LLM calls per step |
| Reliability | Fails silently if one agent's output is malformed | More robust — manager can request a redo |
| Best for | Well-defined, low-ambiguity pipelines | Tasks needing quality control / dynamic delegation |
